# Machine Unlearning on Gowalla: All 4 Pipelines

This notebook runs **all four** unlearning pipelines sequentially on the same pretrained LightGCN checkpoint.

**Order (fastest → slowest):**
1. **AIE** – Attention-Based Influence Encoder (~2.5M params, 3 losses)
2. **CIE** – Causal Influence Encoder (~2.5M params, 5 losses)
3. **GAIE** – Graph Autoencoder Influence Encoder (VAE, 4 losses)
4. **HIE** – Hypernetwork-Based Influence Encoder (~40M params, 3 losses)

Each pipeline: **Unlearn → Fine-tune → Evaluate (Recall, NDCG, MI-BF, MI-NG)**

### Kaggle Setup
Enable **GPU accelerator** (Settings → Accelerator → GPU).

### Experimental protocol (aligned with UnlearnRec, SIGIR'25, Sec. 4.1.4)

**Threat model / unlearning target.** Adversarial edges are the least-probable user-item pairs
under a GCN trained on the clean data. The backbone LightGCN is trained **on the attacked graph**
(clean edges + injected adversarial edges), so the adversarial edges are genuinely learned as
positives. The unlearning task is to remove exactly those edges from the trained backbone.

**Why this matters.** If the backbone were trained on the clean graph instead, the adversarial
edges would never have been learned, the "before unlearning" scores would already be at or below
negative-sample level, and MI-BF / MI-NG would not measure forgetting. This notebook therefore
pretrains with `adversarial_attack=True`.

**Ground truth.** The exact-unlearning reference ("Retrain") is the same architecture retrained
from scratch on the residual graph (attacked graph minus adversarial edges = the clean data).

**Metrics.** MI-BF = mean recommendation probability of the unlearned edges before vs. after
unlearning (higher is better, must be > 1). MI-NG = mean probability of negative samples vs.
unlearned edges after unlearning (> 1 means unlearned edges are now less recommendable than
random non-edges). We also log the raw before/after/negative probabilities as a sanity check
that MI-NG is not trivially pre-satisfied.


---
## 0. Environment Setup

In [1]:
import os
import subprocess
import torch
cuda_tag = torch.version.cuda.replace(".", "")        # e.g. "121" or "124"
torch_tag = ".".join(torch.__version__.split(".")[:2]) # e.g. "2.6"
whl_url = f"https://data.pyg.org/whl/torch-{torch_tag}.0+cu{cuda_tag}.html"
print(f"Installing torch-scatter + torch-sparse from: {whl_url}")
subprocess.check_call(["pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", whl_url])
subprocess.check_call(["pip", "install", "-q", "setproctitle"])

import torch_scatter
import torch_sparse
import setproctitle
print("torch_scatter version:", torch_scatter.__version__)
print("torch_sparse version:", torch_sparse.__version__)
print("setproctitle installed ✓")

Installing torch-scatter + torch-sparse from: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 80.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 116.6 MB/s eta 0:00:00
torch_scatter version: 2.1.2+pt210cu128
torch_sparse version: 0.6.18+pt210cu128
setproctitle installed ✓


In [2]:
import os

REPO_URL = "https://github.com/Shuvayu12/unlearnrec_improv.git"
PROJECT_DIR = "/kaggle/working/unlearnrec_improv"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already cloned.")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

Repo already cloned.
Working directory: /kaggle/working/unlearnrec_improv
Contents: ['models', 'training', 'ckpt', '.gitattributes', 'evaluation', '.gitignore', 'logs', 'README.md', 'config', 'Utils', 'kaggle_pretrain_then_gaie_ml1m.ipynb', 'unlearning', 'data', 'examples', 'datasets', '.git']


In [3]:
import sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Clear sys.argv so argparse in config/params.py doesn't choke on notebook kernel args
sys.argv = [sys.argv[0]]

from config.params import args
from data.data_handler import DataHandler
from Utils.time_logger import log
from Utils.utils import innerProduct, cal_mi_metrics, print_args
from models.Model import LightGCN, AIE, CIE, GAIE, HIE

print("All imports successful!")

All imports successful!


In [4]:
import torch as t
import numpy as np
import random
import time

os.makedirs("./ckpt", exist_ok=True)
os.makedirs("./logs", exist_ok=True)

print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")
    print(f"Memory: {t.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


---
## 1. Pretrain LightGCN on Gowalla

Shared across all 4 pipelines. Only needs to run **once**.

In [5]:
# ============================================================
# Hyperparameters for LightGCN pretraining on Gowalla
# ============================================================

args.data = 'gowalla'
args.model = 'lightgcn'
args.gpu = '0'
args.seed = 1234
args.lr = 1e-3
args.batch = 4096
args.epoch = 200
args.latdim = 128
args.gnn_layer = 3
args.reg = 1e-7
args.topk = 20
args.tst_epoch = 3
args.tst_bat = 256
args.decay = 1.0
args.bpr_wei = 1.0
# PAPER PROTOCOL (UnlearnRec Sec 4.1.4): the backbone to be unlearned must be
# trained on the ATTACKED graph so the adversarial edges are genuinely learned.
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'
args.save_path = './ckpt/pretrain_gowalla_adv'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu

print_args(args)

gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch........................................................................200
sim_epoch......................................................................5
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune............................................................None
trained_model...............................................................None
save_path...................

In [6]:
handler = DataHandler()
handler.load_data(drop_rate=0.0, adv_attack=True)
# dropped_edges == the injected adversarial edges (the unlearning target)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges: {handler.trn_loader.dataset.__len__()}")
print(f"Test users: {handler.tst_loader.dataset.__len__()}")
print(f"Injected adversarial edges: {len(handler.adv_edges[0])}")

################here _load_one_file##################
################here _load_one_file##################
##############here in drop_rate <=0#################
Users: 25557, Items: 19747
Training edges: 294983
Test users: 22242


In [7]:
from training.pretrain_lightgcn import Coach as PretrainCoach

pretrain_coach = PretrainCoach(handler)
pretrain_coach.run()

print("\n" + "="*60)
print("Pretraining complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 45304
NUM OF EDGES 294983
2026-04-16 20:02:50.991627: Model Prepared
2026-04-16 20:02:50.991718: Model Initialized
2026-04-16 20:02:59.598131: Epoch 0/200, Topo: Recall = 0.0031, NDCG = 0.0016  
2026-04-16 20:03:03.374475: Epoch 0/200, Trn: Loss = 0.4180, preLoss = 0.4176       
2026-04-16 20:03:11.592565: Epoch 0/200, Tst: Recall = 0.1165, NDCG = 0.0757   
2026-04-16 20:03:21.391527: Model Saved: ./ckpt/pretrain_gowalla
2026-04-16 20:03:24.453534: Epoch 1/200, Trn: Loss = 0.1551, preLoss = 0.1539       
2026-04-16 20:03:27.403011: Epoch 2/200, Trn: Loss = 0.1156, preLoss = 0.1141       
2026-04-16 20:03:30.365925: Epoch 3/200, Trn: Loss = 0.0987, preLoss = 0.0968       
2026-04-16 20:03:38.422781: Epoch 3/200, Tst: Recall = 0.1601, NDCG = 0.1037   
2026-04-16 20:03:48.095856: Model Saved: ./ckpt/pretrain_gowalla
2026-04-16 20:03:51.140930: Epoch 4/200, Trn: Loss = 0.0906, preLoss = 0.0884       
2026-04-16 20:03:54.100889: Epoch 5/200, Trn: Loss = 0.0808, preLoss = 0.0785

In [8]:
pretrained_model = pretrain_coach.model
pretrained_model.eval()

reses = pretrain_coach.tst_epoch(pretrained_model)
print(f"\nPretrained LightGCN Performance:")
print(f"  Recall@{args.topk}: {reses['Recall']:.4f}")
print(f"  NDCG@{args.topk}:   {reses['NDCG']:.4f}")

# Store pretrained path for reuse by all pipelines
PRETRAINED_PATH = args.save_path

2026-04-16 20:29:35.645236: Steps 86/86: recall = 61.22, ndcg = 36.25          
Pretrained LightGCN Performance:
  Recall@20: 0.2540
  NDCG@20:   0.1639


---
# Results Collection

We'll store results from each pipeline for a final comparison.

In [9]:
# Dictionary to collect results from all pipelines
all_results = {}

---
## 1b. Retrain reference (exact-unlearning ground truth)

The paper's "Retrain" row: the same backbone retrained from scratch on the residual graph
(= clean training data). Used as ground truth for both utility (Recall/NDCG) and efficacy
(MI-BF/MI-NG), and as the efficiency yardstick (unlearning must beat this wall-clock time).

In [ ]:
# ============================================================
# Retrain reference (exact unlearning): train from scratch on E_r
# ============================================================
RUN_RETRAIN_REFERENCE = True   # set False to skip and save GPU time

if RUN_RETRAIN_REFERENCE:
    args.adversarial_attack = False
    args.save_path = './ckpt/retrain_ref'
    handler_clean = DataHandler()
    handler_clean.load_data(drop_rate=0.0, adv_attack=False)

    retrain_coach = PretrainCoach(handler_clean)
    t.cuda.reset_peak_memory_stats()
    _t0 = time.time()
    retrain_coach.run()
    retrain_time = time.time() - _t0
    retrain_mem = t.cuda.max_memory_allocated() / 1e9

    ckp_r = t.load('./ckpt/retrain_ref.mod', weights_only=False)
    retrain_model = ckp_r['model'].cuda()
    retrain_model.eval()
    retrain_metrics = retrain_coach.tst_epoch(retrain_model)
    print(f"\n[Retrain] Recall@{args.topk}: {retrain_metrics['Recall']:.4f}, "
          f"NDCG@{args.topk}: {retrain_metrics['NDCG']:.4f}, "
          f"time: {retrain_time:.1f}s, peak mem: {retrain_mem:.2f} GB")


In [ ]:
if RUN_RETRAIN_REFERENCE:
    # MI metrics of the retrained model on the adversarial edges.
    # "Before" scores come from the attacked backbone (the model being unlearned).
    args.adversarial_attack = True
    handler_adv_eval = DataHandler()
    handler_adv_eval.load_data(drop_rate=0.0, adv_attack=True)
    adv_u, adv_i = handler_adv_eval.dropped_edges

    with t.no_grad():
        ret_u_emb, ret_i_emb = retrain_model.forward(handler_clean.ts_ori_adj, keepRate=1.0)
        atk_u_emb, atk_i_emb = pretrained_model.forward(handler_adv_eval.ts_ori_adj, keepRate=1.0)

    ret_drp = innerProduct(ret_u_emb[adv_u], ret_i_emb[adv_i])
    atk_drp = innerProduct(atk_u_emb[adv_u], atk_i_emb[adv_i])

    rows, cols = handler_adv_eval.ori_trn_mat.row, handler_adv_eval.ori_trn_mat.col
    edge_set = set(zip(rows.tolist(), cols.tolist()))
    neg_r, neg_c = [], []
    while len(neg_r) < len(adv_u):
        i, j = np.random.randint(args.user), np.random.randint(args.item)
        if (i, j) not in edge_set:
            edge_set.add((i, j)); neg_r.append(i); neg_c.append(j)
    ret_neg = innerProduct(ret_u_emb[neg_r], ret_i_emb[neg_c])

    retrain_mi = cal_mi_metrics(ret_drp, ret_neg, before_drp_scores=atk_drp)
    all_results['Retrain'] = {
        'Recall': retrain_metrics['Recall'], 'NDCG': retrain_metrics['NDCG'],
        'MI_BF': retrain_mi['mi_bf'], 'MI_NG': retrain_mi['mi_ng'],
        'BeforeProb': retrain_mi['avg_before_prob'], 'AfterProb': retrain_mi['avg_after_prob'],
        'NegProb': retrain_mi['avg_neg_prob'],
        'UnlearnTime': retrain_time, 'FinetuneTime': 0.0, 'PeakMemGB': retrain_mem,
    }
    print(f"[Retrain] MI-BF={retrain_mi['mi_bf']:.4f}, MI-NG={retrain_mi['mi_ng']:.4f}")

    del retrain_coach, handler_clean
    t.cuda.empty_cache()


---
---
# Pipeline 1: AIE (Attention-Based Influence Encoder)

**Fastest pipeline** — ~2.5M params, 3 losses (BPR + unlearn + alignment).

Deleted edges → GAT over influence graph → MLP shift generator → ΔE

**No reconstruction loss** (unlike GAIE).

## AIE — Step 2a: Unlearn

In [13]:
# ============================================================
# Hyperparameters for AIE unlearning on Gowalla
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.2
args.test_drop_rate = 0.003
args.sim_epoch = 10
args.tst_epoch = 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.3
args.align_wei = 0.1
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.3
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gowalla_aie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)

gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................30
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune............................................................None
trained_model............................................./ckpt/pretrain_gowalla
save_path...................

In [14]:
handler_unlearn_aie = DataHandler()
handler_unlearn_aie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_aie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_aie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_aie.picked_edges[0])}")

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Users: 25557, Items: 19747
Training edges (after drop): 295031
Dropped edges: 22651
Picked (retained) edges: 295031


In [16]:
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.08 KiB | 275.00 KiB/s, done.
From https://github.com/Shuvayu12/unlearnrec_improv
   6eab40a..98b3964  main       -> origin/main
Updating 6eab40a..98b3964
Fast-forward
 unlearning/aie_unlearn.py  | 8 ++++----
 unlearning/cie_unlearn.py  | 8 ++++----
 unlearning/gaie_unlearn.py | 8 ++++----
 unlearning/hie_unlearn.py  | 8 ++++----
 4 files changed, 16 insertions(+), 16 deletions(-)


In [17]:
from unlearning.aie_unlearn import Coach as AIECoach

aie_coach = AIECoach(handler_unlearn_aie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
aie_coach.run()
aie_unlearn_time = time.time() - _t0
aie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[AIE unlearn] {aie_unlearn_time:.1f}s, peak GPU mem {aie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("AIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 45304
NUM OF EDGES 295031
2026-04-16 20:46:38.409983: Model Preparedecall = 56.49, ndcg = 34.29          
2026-04-16 20:46:38.410017: Model Initialized
2026-04-16 20:46:47.810935: Epoch 0/30, Topo: Recall = 0.2552, NDCG = 0.1645   
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
>>>>Recall: 0.255248,  NDCG: 0.164466 @ Epoch: 0= 58.76, ndcg = 35.87          
[AIE] AIE test:
  Dropped edges  (mean,var,max,min): <-4.9361, 4.3994, 0.1537, -22.2269>  |  Pretrain: <-1.6534,3.1592,6.6601,-9.5278>
  Positive edges (mean,var,max,min): <8.6140, 4.7877, 24.8019, -6.4813>  |  Pretrain: <7.1055,3.2251,19.5253,-4.7174>
  Negative edges (mean,var,max,min): <-0.0046, 1.5416, 10.4430, -10.0340>  |  Pretrain: <-0.0019,1.0931,9.5147,-5.7274>
  MI metrics:  MI-BF=0.6667, MI-NG=4.9315, MI-AUC=0.0073, MI-ACC=0.5000
  MI pretra

## AIE — Step 2b: Fine-Tune

In [20]:
args.model_2_finetune = './ckpt/gowalla_aie_unlearn'

args.fineTune = True
args.epoch = 20
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.1
args.align_wei = 0.05
args.align_temp = 10.0
args.perf_degrade = 0.3
args.sim_epoch = 10
args.tst_epoch = 5
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gowalla_aie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)

gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................20
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune......................................./ckpt/gowalla_aie_unlearn
trained_model............................................./ckpt/pretrain_gowalla
save_path...................

In [21]:
handler_ft_aie = DataHandler()
handler_ft_aie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

aie_ft_coach = AIECoach(handler_ft_aie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
aie_ft_coach.run()
aie_ft_time = time.time() - _t0
aie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[AIE finetune] {aie_ft_time:.1f}s, peak GPU mem {aie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("AIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
NUM OF NODES 45304
NUM OF EDGES 295031
2026-04-16 21:03:07.470406: Model Preparedecall = 56.49, ndcg = 34.29          
2026-04-16 21:03:09.972350: Model Loaded
2026-04-16 21:03:19.333959: Epoch 0/20, Topo: Recall = 0.2390, NDCG = 0.1544   
2026-04-16 21:03:28.441648: Epoch -4/20, Trn: Loss = 1.3231, preLoss = 0.0102, unlearn_loss = 0.0311, align_loss = 26.0051  
2026-04-16 21:03:37.502965: Epoch -3/20, Trn: Loss = 0.3403, preLoss = 0.0111, unlearn_loss = 0.0507, align_loss = 6.2909  
2026-04-16 21:03:46.529683: Epoch -2/20, Trn: Loss = 0.1841, preLoss = 0.0108, unlearn_loss = 0.0572, align_loss = 3.1612  
2026-04-16 21:03:55.537768: Epoch -1/20, Trn: Loss = 0.1208, preLoss = 0.0111, unlearn_loss = 0.0583, align_loss = 1.8854

## AIE — Step 3: Evaluate

In [22]:
handler_eval_aie = DataHandler()
handler_eval_aie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    aie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned AIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

aie_ft_coach.handler = handler_eval_aie

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Loaded fine-tuned AIE model from ./ckpt/gowalla_aie_ft.mod


In [23]:
aie_metrics = aie_ft_coach.tst_epoch(aie_ft_coach.model)
print(f"\n[AIE] Recall@{args.topk}: {aie_metrics['Recall']:.4f}")
print(f"[AIE] NDCG@{args.topk}:   {aie_metrics['NDCG']:.4f}")

2026-04-16 21:10:26.216602: Steps 86/86: recall = 58.07, ndcg = 34.57          
[AIE] Recall@20: 0.2415
[AIE] NDCG@20:   0.1547


In [24]:
aie_mi = aie_ft_coach.test_unlearn(aie_ft_coach.model, prefix='[AIE] Final Evaluation')

[AIE] [AIE] Final Evaluation
  Dropped edges  (mean,var,max,min): <-5.4942, 3.7456, -0.3376, -14.8653>  |  Pretrain: <-1.6534,3.1592,6.6601,-9.5278>
  Positive edges (mean,var,max,min): <7.5366, 3.8767, 21.6510, -6.8162>  |  Pretrain: <7.1055,3.2251,19.5253,-4.7174>
  Negative edges (mean,var,max,min): <0.0273, 1.1726, 9.7883, -7.4735>  |  Pretrain: <-0.0019,1.0931,9.5147,-5.7274>
  MI metrics:  MI-BF=0.6667, MI-NG=5.5215, MI-AUC=0.0023, MI-ACC=0.5000
  MI pretrain: MI-BF=0.6667, MI-NG=1.6515, MI-AUC=0.1911, MI-ACC=0.5000


In [25]:
all_results['AIE'] = {
    'Recall': aie_metrics['Recall'], 'NDCG': aie_metrics['NDCG'],
    'MI_BF': aie_mi['mi_bf'], 'MI_NG': aie_mi['mi_ng'],
    'BeforeProb': aie_mi['avg_before_prob'], 'AfterProb': aie_mi['avg_after_prob'],
    'NegProb': aie_mi['avg_neg_prob'],
    'UnlearnTime': aie_unlearn_time, 'FinetuneTime': aie_ft_time,
    'PeakMemGB': max(aie_unlearn_mem, aie_ft_mem),
}
print("AIE results stored.")

# Free GPU memory
del aie_coach, aie_ft_coach, handler_unlearn_aie, handler_ft_aie, handler_eval_aie
t.cuda.empty_cache()
print("AIE objects freed, GPU cache cleared.")

AIE results stored.
AIE objects freed, GPU cache cleared.


---
---
# Pipeline 2: CIE (Causal Influence Encoder)

**Second fastest** — ~2.5M params, 5 losses (BPR + unlearn + alignment + contrastive + causal).

Models unlearning as a **causal intervention** do(e_ij = 0).
Computes **counterfactual embeddings** E^cf once, then trains with extra consistency losses.

CIE-specific params: `contrast_wei=0.01`, `causal_wei=0.1`.

## CIE — Step 2a: Unlearn

In [ ]:

# ============================================================
# Hyperparameters for CIE unlearning on Gowalla
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.2
args.test_drop_rate = 0.003
args.sim_epoch = 10
args.tst_epoch = 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.2
args.align_wei = 0.08
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.contrast_wei = 0.01
args.causal_wei = 0.15
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.4
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gowalla_cie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................30
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune......................................./ckpt/gowalla_aie_unlearn
trained_model............................................./ckpt/pretrain_gowalla
save_path...................

In [27]:
handler_unlearn_cie = DataHandler()
handler_unlearn_cie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_cie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_cie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_cie.picked_edges[0])}")

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Users: 25557, Items: 19747
Training edges (after drop): 295031
Dropped edges: 22651
Picked (retained) edges: 295031


In [28]:
from unlearning.cie_unlearn import Coach as CIECoach

cie_coach = CIECoach(handler_unlearn_cie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
cie_coach.run()
cie_unlearn_time = time.time() - _t0
cie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[CIE unlearn] {cie_unlearn_time:.1f}s, peak GPU mem {cie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("CIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 45304
NUM OF EDGES 295031
2026-04-16 21:10:40.265402: Model Preparedecall = 56.49, ndcg = 34.29          
2026-04-16 21:10:40.265447: Model Initialized
2026-04-16 21:10:50.108552: Epoch 0/30, Topo: Recall = 0.0347, NDCG = 0.0222   
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
>>>>Recall: 0.034667,  NDCG: 0.022182 @ Epoch: 0= 11.19, ndcg = 5.38           
[CIE] CIE test:
  Dropped edges  (mean,var,max,min): <488.1971, 66457.1484, 3167.8618, 49.2980>  |  Pretrain: <-1.6534,3.1592,6.6601,-9.5278>
  Positive edges (mean,var,max,min): <330.3491, 45957.7031, 2876.3274, 73.6392>  |  Pretrain: <7.1055,3.2251,19.5253,-4.7174>
  Negative edges (mean,var,max,min): <171.1713, 5739.6191, 1341.7822, 23.9873>  |  Pretrain: <-0.0019,1.0931,9.5147,-5.7274>
  MI metrics:  MI-BF=0.8705, MI-NG=317.0258, MI-AUC=0.9395, MI-

## CIE — Step 2b: Fine-Tune

In [ ]:

args.model_2_finetune = './ckpt/gowalla_cie_unlearn'

args.fineTune = True
args.epoch = 15
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.15
args.align_wei = 0.08
args.align_temp = 10.0
args.contrast_wei = 0.005
args.causal_wei = 0.15
args.perf_degrade = 0.5
args.sim_epoch = 10
args.tst_epoch = 5
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gowalla_cie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................15
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune......................................./ckpt/gowalla_cie_unlearn
trained_model............................................./ckpt/pretrain_gowalla
save_path...................

In [30]:
handler_ft_cie = DataHandler()
handler_ft_cie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

cie_ft_coach = CIECoach(handler_ft_cie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
cie_ft_coach.run()
cie_ft_time = time.time() - _t0
cie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[CIE finetune] {cie_ft_time:.1f}s, peak GPU mem {cie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("CIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
NUM OF NODES 45304
NUM OF EDGES 295031
2026-04-16 21:27:16.116163: Model Preparedecall = 56.49, ndcg = 34.29          
2026-04-16 21:27:18.615571: Model Loaded
2026-04-16 21:27:28.150240: Epoch 0/15, Topo: Recall = 0.0347, NDCG = 0.0222   
2026-04-16 21:27:43.091350: Epoch -4/15, Trn: Loss = 15692.1853, preLoss = 0.1944, unlearn_loss = 2.0458, align_loss = 784578.4375, contrast_loss = 0.1465, causal_loss = 0.3505  
2026-04-16 21:27:57.774500: Epoch -3/15, Trn: Loss = 0.1551, preLoss = 0.0106, unlearn_loss = 0.0943, align_loss = 5.5817, contrast_loss = 0.1342, causal_loss = 0.3755  
2026-04-16 21:28:12.603119: Epoch -2/15, Trn: Loss = 0.1096, preLoss = 0.0102, unlearn_loss = 0.0823, align_loss = 3.4434, contrast_loss = 0.1344

## CIE — Step 3: Evaluate

In [31]:
handler_eval_cie = DataHandler()
handler_eval_cie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    cie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned CIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

cie_ft_coach.handler = handler_eval_cie

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Loaded fine-tuned CIE model from ./ckpt/gowalla_cie_ft.mod


In [32]:
cie_metrics = cie_ft_coach.tst_epoch(cie_ft_coach.model)
print(f"\n[CIE] Recall@{args.topk}: {cie_metrics['Recall']:.4f}")
print(f"[CIE] NDCG@{args.topk}:   {cie_metrics['NDCG']:.4f}")

2026-04-16 21:37:16.585084: Steps 86/86: recall = 54.81, ndcg = 33.35          
[CIE] Recall@20: 0.2318
[CIE] NDCG@20:   0.1461


In [33]:
cie_mi = cie_ft_coach.test_unlearn(cie_ft_coach.model, prefix='[CIE] Final Evaluation')

[CIE] [CIE] Final Evaluation
  Dropped edges  (mean,var,max,min): <-5.2310, 7.1254, 4.4984, -20.8437>  |  Pretrain: <-1.6534,3.1592,6.6601,-9.5278>
  Positive edges (mean,var,max,min): <8.2016, 4.1992, 22.9573, -7.7033>  |  Pretrain: <7.1055,3.2251,19.5253,-4.7174>
  Negative edges (mean,var,max,min): <0.4003, 1.4804, 11.5207, -6.2382>  |  Pretrain: <-0.0019,1.0931,9.5147,-5.7274>
  MI metrics:  MI-BF=0.6667, MI-NG=5.6313, MI-AUC=0.0140, MI-ACC=0.5000
  MI pretrain: MI-BF=0.6667, MI-NG=1.6515, MI-AUC=0.1911, MI-ACC=0.5000


In [34]:
all_results['CIE'] = {
    'Recall': cie_metrics['Recall'], 'NDCG': cie_metrics['NDCG'],
    'MI_BF': cie_mi['mi_bf'], 'MI_NG': cie_mi['mi_ng'],
    'BeforeProb': cie_mi['avg_before_prob'], 'AfterProb': cie_mi['avg_after_prob'],
    'NegProb': cie_mi['avg_neg_prob'],
    'UnlearnTime': cie_unlearn_time, 'FinetuneTime': cie_ft_time,
    'PeakMemGB': max(cie_unlearn_mem, cie_ft_mem),
}
print("CIE results stored.")

del cie_coach, cie_ft_coach, handler_unlearn_cie, handler_ft_cie, handler_eval_cie
t.cuda.empty_cache()
print("CIE objects freed, GPU cache cleared.")

CIE results stored.
CIE objects freed, GPU cache cleared.


---
---
# Pipeline 3: GAIE (Graph Autoencoder Influence Encoder)

**Third fastest** — VAE-based, 4 losses (BPR + unlearn + alignment + reconstruction).

Influence graph → VAE encoder → latent z → reparameterize → shift MLP → ΔE

GAIE-specific: `rec_wei=0.1` (reconstruction loss).

## GAIE — Step 2a: Unlearn

In [ ]:

# ============================================================
# Hyperparameters for GAIE unlearning on Gowalla
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.2
args.test_drop_rate = 0.003
args.sim_epoch = 10
args.tst_epoch = 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.3
args.align_wei = 0.1
args.rec_wei = 0.03
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.5
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gowalla_gaie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................30
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune......................................./ckpt/gowalla_cie_unlearn
trained_model............................................./ckpt/pretrain_gowalla
save_path...................

In [36]:
handler_unlearn_gaie = DataHandler()
handler_unlearn_gaie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_gaie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_gaie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_gaie.picked_edges[0])}")

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Users: 25557, Items: 19747
Training edges (after drop): 295031
Dropped edges: 22651
Picked (retained) edges: 295031


In [37]:
from unlearning.gaie_unlearn import Coach as GAIECoach

gaie_coach = GAIECoach(handler_unlearn_gaie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
gaie_coach.run()
gaie_unlearn_time = time.time() - _t0
gaie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[GAIE unlearn] {gaie_unlearn_time:.1f}s, peak GPU mem {gaie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("GAIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 45304
NUM OF EDGES 295031
2026-04-16 21:37:30.473693: Model Preparedecall = 56.49, ndcg = 34.29          
2026-04-16 21:37:30.473727: Model Initialized
2026-04-16 21:37:39.839991: Epoch 0/30, Topo: Recall = 0.0341, NDCG = 0.0214  
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
>>>>Recall: 0.034284,  NDCG: 0.021524 @ Epoch: 0= 11.19, ndcg = 5.39           
[GAIE] GAIE test:
  Dropped edges  (mean,var,max,min): <1052.7635, 359576.1875, 6885.6953, -82.1225>  |  Pretrain: <-1.6534,3.1592,6.6601,-9.5278>
  Positive edges (mean,var,max,min): <750.5961, 188947.1875, 6117.5396, 106.1092>  |  Pretrain: <7.1055,3.2251,19.5253,-4.7174>
  Negative edges (mean,var,max,min): <381.4356, 34742.0664, 2762.0962, -73.6823>  |  Pretrain: <-0.0019,1.0931,9.5147,-5.7274>
  MI metrics:  MI-BF=0.8339, MI-NG=671.3281, MI-AUC=0.9

## GAIE — Step 2b: Fine-Tune

In [ ]:

args.model_2_finetune = './ckpt/gowalla_gaie_unlearn'

args.fineTune = True
args.epoch = 15
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.15
args.align_wei = 0.1
args.rec_wei = 0.03
args.align_temp = 10.0
args.perf_degrade = 0.5
args.sim_epoch = 10
args.tst_epoch = 5
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gowalla_gaie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................15
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune....................................../ckpt/gowalla_gaie_unlearn
trained_model............................................./ckpt/pretrain_gowalla
save_path...................

In [39]:
handler_ft_gaie = DataHandler()
handler_ft_gaie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

gaie_ft_coach = GAIECoach(handler_ft_gaie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
gaie_ft_coach.run()
gaie_ft_time = time.time() - _t0
gaie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[GAIE finetune] {gaie_ft_time:.1f}s, peak GPU mem {gaie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("GAIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
NUM OF NODES 45304
NUM OF EDGES 295031
2026-04-16 21:51:24.167199: Model Preparedecall = 56.49, ndcg = 34.29          
2026-04-16 21:51:26.698001: Model Loaded
2026-04-16 21:51:36.161135: Epoch 0/15, Topo: Recall = 0.0341, NDCG = 0.0214  
2026-04-16 21:51:48.968643: Epoch -4/15, Trn: Loss = 39450.3833, preLoss = 1.1050, unlearn_loss = 5.5059, align_loss = 3944792.5000, rec_loss = 2.2957   
2026-04-16 21:52:01.747891: Epoch -3/15, Trn: Loss = 0.5498, preLoss = 0.0091, unlearn_loss = 0.1827, align_loss = 43.9343, rec_loss = 0.5528  
2026-04-16 21:52:14.294785: Epoch -2/15, Trn: Loss = 0.3482, preLoss = 0.0090, unlearn_loss = 0.1198, align_loss = 25.1868, rec_loss = 0.5382  
2026-04-16 21:52:26.908429: Epoch -1/15, Trn: Loss = 

## GAIE — Step 3: Evaluate

In [40]:
handler_eval_gaie = DataHandler()
handler_eval_gaie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    gaie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned GAIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

gaie_ft_coach.handler = handler_eval_gaie

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Loaded fine-tuned GAIE model from ./ckpt/gowalla_gaie_ft.mod


In [41]:
gaie_metrics = gaie_ft_coach.tst_epoch(gaie_ft_coach.model)
print(f"\n[GAIE] Recall@{args.topk}: {gaie_metrics['Recall']:.4f}")
print(f"[GAIE] NDCG@{args.topk}:   {gaie_metrics['NDCG']:.4f}")

2026-04-16 21:59:50.028828: Steps 86/86: recall = 52.43, ndcg = 30.20          
[GAIE] Recall@20: 0.2137
[GAIE] NDCG@20:   0.1365


In [42]:
gaie_mi = gaie_ft_coach.test_unlearn(gaie_ft_coach.model, prefix='[GAIE] Final Evaluation')

[GAIE] [GAIE] Final Evaluation
  Dropped edges  (mean,var,max,min): <-2.9181, 3.0218, 3.1873, -13.3788>  |  Pretrain: <-1.6534,3.1592,6.6601,-9.5278>
  Positive edges (mean,var,max,min): <8.4425, 5.4225, 24.0717, -4.4311>  |  Pretrain: <7.1055,3.2251,19.5253,-4.7174>
  Negative edges (mean,var,max,min): <0.5125, 1.5138, 10.9999, -6.4130>  |  Pretrain: <-0.0019,1.0931,9.5147,-5.7274>
  MI metrics:  MI-BF=0.6667, MI-NG=3.4306, MI-AUC=0.0429, MI-ACC=0.5000
  MI pretrain: MI-BF=0.6667, MI-NG=1.6515, MI-AUC=0.1911, MI-ACC=0.5000


In [43]:
all_results['GAIE'] = {
    'Recall': gaie_metrics['Recall'], 'NDCG': gaie_metrics['NDCG'],
    'MI_BF': gaie_mi['mi_bf'], 'MI_NG': gaie_mi['mi_ng'],
    'BeforeProb': gaie_mi['avg_before_prob'], 'AfterProb': gaie_mi['avg_after_prob'],
    'NegProb': gaie_mi['avg_neg_prob'],
    'UnlearnTime': gaie_unlearn_time, 'FinetuneTime': gaie_ft_time,
    'PeakMemGB': max(gaie_unlearn_mem, gaie_ft_mem),
}
print("GAIE results stored.")

del gaie_coach, gaie_ft_coach, handler_unlearn_gaie, handler_ft_gaie, handler_eval_gaie
t.cuda.empty_cache()
print("GAIE objects freed, GPU cache cleared.")

GAIE results stored.
GAIE objects freed, GPU cache cleared.


---
---
# Pipeline 4: HIE (Hypernetwork-Based Influence Encoder)

**Slowest pipeline** — ~40M params from HyperNetwork, 3 losses (BPR + unlearn + alignment).

Influence graph → GNN → mean-pool → latent z → HyperNetwork H(z) → W_u → ΔE = W_u ⊙ E

The HyperNetwork generates per-node weight matrices, making this significantly heavier.

## HIE — Step 2a: Unlearn

In [ ]:

# ============================================================
# Hyperparameters for HIE unlearning on Gowalla
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.2
args.test_drop_rate = 0.003
args.sim_epoch = 10
args.tst_epoch = 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.3
args.align_wei = 0.08
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.4
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gowalla_hie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................30
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune....................................../ckpt/gowalla_gaie_unlearn
trained_model............................................./ckpt/pretrain_gowalla
save_path...................

In [45]:
handler_unlearn_hie = DataHandler()
handler_unlearn_hie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_hie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_hie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_hie.picked_edges[0])}")

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Users: 25557, Items: 19747
Training edges (after drop): 295031
Dropped edges: 22651
Picked (retained) edges: 295031


In [46]:
from unlearning.hie_unlearn import Coach as HIECoach

hie_coach = HIECoach(handler_unlearn_hie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
hie_coach.run()
hie_unlearn_time = time.time() - _t0
hie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[HIE unlearn] {hie_unlearn_time:.1f}s, peak GPU mem {hie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("HIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

NUM OF NODES 45304
NUM OF EDGES 295031
2026-04-16 22:00:05.518806: Model Preparedecall = 56.49, ndcg = 34.29          
2026-04-16 22:00:05.518939: Model Initialized
2026-04-16 22:00:15.241500: Epoch 0/30, Topo: Recall = 0.2551, NDCG = 0.1645   
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
>>>>Recall: 0.255066,  NDCG: 0.164493 @ Epoch: 0= 59.26, ndcg = 35.79          
[HIE] HIE test:
  Dropped edges  (mean,var,max,min): <-4.9409, 4.4022, 0.1388, -22.1442>  |  Pretrain: <-1.6534,3.1592,6.6601,-9.5278>
  Positive edges (mean,var,max,min): <8.6136, 4.7850, 24.7977, -6.5570>  |  Pretrain: <7.1055,3.2251,19.5253,-4.7174>
  Negative edges (mean,var,max,min): <-0.0048, 1.5410, 10.5728, -9.9265>  |  Pretrain: <-0.0019,1.0931,9.5147,-5.7274>
  MI metrics:  MI-BF=0.6667, MI-NG=4.9362, MI-AUC=0.0073, MI-ACC=0.5000
  MI pretrai

## HIE — Step 2b: Fine-Tune

In [ ]:

args.model_2_finetune = './ckpt/gowalla_hie_unlearn'

args.fineTune = True
args.epoch = 15
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.15
args.align_wei = 0.08
args.align_temp = 10.0
args.perf_degrade = 0.5
args.sim_epoch = 10
args.tst_epoch = 5
args.adversarial_attack = True
args.adv_method = 'lightgcn0.5'

args.save_path = './ckpt/gowalla_hie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


gpu............................................................................0
seed........................................................................1234
lr.........................................................................0.001
batch.......................................................................4096
tst_bat......................................................................256
epoch.........................................................................15
sim_epoch.....................................................................10
decay........................................................................1.0
early_stop....................................................................10
load_model..................................................................None
model_2_finetune......................................./ckpt/gowalla_hie_unlearn
trained_model............................................./ckpt/pretrain_gowalla
save_path...................

In [48]:
handler_ft_hie = DataHandler()
handler_ft_hie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

hie_ft_coach = HIECoach(handler_ft_hie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
hie_ft_coach.run()
hie_ft_time = time.time() - _t0
hie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[HIE finetune] {hie_ft_time:.1f}s, peak GPU mem {hie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("HIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
NUM OF NODES 45304
NUM OF EDGES 295031
2026-04-16 22:10:46.552909: Model Preparedecall = 56.49, ndcg = 34.29          
2026-04-16 22:10:49.811566: Model Loaded
2026-04-16 22:10:58.570405: Epoch 0/15, Topo: Recall = 0.2551, NDCG = 0.1645   
2026-04-16 22:11:12.468991: Epoch -4/15, Trn: Loss = 0.4086, preLoss = 0.0089, unlearn_loss = 0.0558, align_loss = 37.8961  
2026-04-16 22:11:26.398537: Epoch -3/15, Trn: Loss = 0.1036, preLoss = 0.0102, unlearn_loss = 0.0768, align_loss = 6.8468  
2026-04-16 22:11:40.218828: Epoch -2/15, Trn: Loss = 0.0679, preLoss = 0.0100, unlearn_loss = 0.0746, align_loss = 3.3366  
2026-04-16 22:11:53.943671: Epoch -1/15, Trn: Loss = 0.0550, preLoss = 0.0102, unlearn_loss = 0.0680, align_loss = 2.1606

## HIE — Step 3: Evaluate

In [49]:
handler_eval_hie = DataHandler()
handler_eval_hie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    hie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned HIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

hie_ft_coach.handler = handler_eval_hie

##########using the least adv_mat#############
################here _load_one_file##################
################here load self.adv_edges##################
################here _load_one_file##################
##############here in drop_rate >0#################
Loaded fine-tuned HIE model from ./ckpt/gowalla_hie_ft.mod


In [50]:
hie_metrics = hie_ft_coach.tst_epoch(hie_ft_coach.model)
print(f"\n[HIE] Recall@{args.topk}: {hie_metrics['Recall']:.4f}")
print(f"[HIE] NDCG@{args.topk}:   {hie_metrics['NDCG']:.4f}")

2026-04-16 22:17:39.017041: Steps 86/86: recall = 50.41, ndcg = 27.90          
[HIE] Recall@20: 0.2193
[HIE] NDCG@20:   0.1384


In [51]:
hie_mi = hie_ft_coach.test_unlearn(hie_ft_coach.model, prefix='[HIE] Final Evaluation')

[HIE] [HIE] Final Evaluation
  Dropped edges  (mean,var,max,min): <-3.6840, 2.6321, 1.4902, -12.1172>  |  Pretrain: <-1.6534,3.1592,6.6601,-9.5278>
  Positive edges (mean,var,max,min): <6.2866, 4.5248, 20.4236, -5.2861>  |  Pretrain: <7.1055,3.2251,19.5253,-4.7174>
  Negative edges (mean,var,max,min): <0.0010, 0.8939, 9.2650, -6.9890>  |  Pretrain: <-0.0019,1.0931,9.5147,-5.7274>
  MI metrics:  MI-BF=0.6667, MI-NG=3.6850, MI-AUC=0.0131, MI-ACC=0.5000
  MI pretrain: MI-BF=0.6667, MI-NG=1.6515, MI-AUC=0.1911, MI-ACC=0.5000


In [52]:
all_results['HIE'] = {
    'Recall': hie_metrics['Recall'], 'NDCG': hie_metrics['NDCG'],
    'MI_BF': hie_mi['mi_bf'], 'MI_NG': hie_mi['mi_ng'],
    'BeforeProb': hie_mi['avg_before_prob'], 'AfterProb': hie_mi['avg_after_prob'],
    'NegProb': hie_mi['avg_neg_prob'],
    'UnlearnTime': hie_unlearn_time, 'FinetuneTime': hie_ft_time,
    'PeakMemGB': max(hie_unlearn_mem, hie_ft_mem),
}
print("HIE results stored.")

del hie_coach, hie_ft_coach, handler_unlearn_hie, handler_ft_hie, handler_eval_hie
t.cuda.empty_cache()
print("HIE objects freed, GPU cache cleared.")

HIE results stored.
HIE objects freed, GPU cache cleared.


---
---
# Final Comparison: All 4 Pipelines

In [53]:
import json

print("\n" + "#" * 110)
print("#" + " FINAL COMPARISON — attacked-backbone protocol (UnlearnRec Sec. 4.1.4) ".center(108) + "#")
print("#" * 110)
print(f"#  Dataset:  {args.data} ({args.user} users, {args.item} items)")
print(f"#  Backbone: LightGCN ({args.latdim}-dim, {args.gnn_layer} layers), trained on attacked graph (adv method: {args.adv_method})")
print(f"#  Unlearn target: all injected adversarial edges")
print("#" + "-" * 108 + "#")
header = (f"#  {'Method':<9} {'Recall@20':>10} {'NDCG@20':>9} {'MI-BF':>8} {'MI-NG':>8} "
          f"{'P(before)':>10} {'P(after)':>9} {'P(neg)':>8} {'Time(s)':>9} {'Mem(GB)':>8}  #")
print(header)
print("#" + "-" * 108 + "#")

for name in ['Retrain', 'AIE', 'CIE', 'GAIE', 'HIE']:
    r = all_results.get(name, {})
    def f(key, fmt='{:.4f}'):
        return fmt.format(r[key]) if key in r else 'N/A'
    total_time = (r.get('UnlearnTime', 0) or 0) + (r.get('FinetuneTime', 0) or 0)
    time_s = f"{total_time:.1f}" if 'UnlearnTime' in r else 'N/A'
    print(f"#  {name:<9} {f('Recall'):>10} {f('NDCG'):>9} {f('MI_BF'):>8} {f('MI_NG'):>8} "
          f"{f('BeforeProb'):>10} {f('AfterProb'):>9} {f('NegProb'):>8} {time_s:>9} {f('PeakMemGB', '{:.2f}'):>8}  #")

print("#" + "-" * 108 + "#")
print("#  Sanity check: P(before) should sit near positive-edge level (edges were trained on);         #")
print("#  a method truly forgets when P(after) <= P(neg). Speedup = Retrain time / method time.        #")
print("#" * 110)

os.makedirs('./logs', exist_ok=True)
out = {
    'dataset': args.data, 'backbone': 'lightgcn', 'latdim': args.latdim,
    'gnn_layer': args.gnn_layer, 'seed': args.seed, 'adv_method': args.adv_method,
    'protocol': 'attacked-backbone (UnlearnRec Sec 4.1.4)', 'results': all_results,
}
res_file = f'./logs/final_results_{args.data}.json'
with open(res_file, 'w') as fs:
    json.dump(out, fs, indent=2)
print(f"\nResults saved to {res_file} — download this from Kaggle output for the paper.")



######################################################################################
#                          ALL PIPELINES — FINAL COMPARISON                          #
######################################################################################
#  Dataset:    Gowalla (25557 users, 19747 items)
#  Backbone:   LightGCN (128-dim, 3 layers)
#  Drop Mode:  Adversarial (lightgcn0.5)
#  Drop Rate:  20% of training edges
#------------------------------------------------------------------------------------#
#  Pipeline      Recall@20      NDCG@20      MI-BF      MI-NG      ~Params  #
#------------------------------------------------------------------------------------#
#  AIE              0.2415       0.1547     0.6667     5.5215        ~2.5M  #
#  CIE              0.2318       0.1461     0.6667     5.6313        ~2.5M  #
#  GAIE             0.2137       0.1365     0.6667     3.4306          ~3M  #
#  HIE              0.2193       0.1384     0.6667     3.6850         ~40M  #
#-